# Optimasi Parameter Moving Average (MA)
Notebook ini digunakan untuk mencari **Window Size** terbaik untuk metode Moving Average pada setiap bandara.

**Konfigurasi:**
- Data mulai tahun 2015.
- Menghapus data periode Covid-19 (2020-2022).
- Mencari Window terbaik antara 2 hingga 24 bulan.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

file_path = 'target_penumpang_domestik_internasional_2006_2024.xlsx'
df_raw = pd.read_excel(file_path)

bulan_map = {
    'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'Mei':5, 'Jun':6,
    'Jul':7, 'Agu':8, 'Sep':9, 'Okt':10, 'Nov':11, 'Des':12
}
df_raw['Bulan_num'] = df_raw['Bulan'].map(bulan_map)
df_raw['Tanggal'] = pd.to_datetime(df_raw['Tahun'].astype(str) + '-' + df_raw['Bulan_num'].astype(str) + '-01')
df_raw = df_raw.sort_values('Tanggal').reset_index(drop=True)

df = df_raw[df_raw['Tahun'] >= 2015].reset_index(drop=True)
df = df[~df['Tahun'].isin([2020, 2021, 2022])].reset_index(drop=True)

print(f"Data siap. Range: {df['Tanggal'].min()} sampai {df['Tanggal'].max()}")

Data siap. Range: 2015-01-01 00:00:00 sampai 2025-12-01 00:00:00


In [ ]:
bandara_cols = [
    'soekarno_hatta_jakarta_domestik', 'ngurah_rai_bali_domestik', 
    'juanda_surabaya_domestik', 'kualanamu_medan_domestik', 
    'hasanudin_makassar_domestik', 'soekarno_hatta_jakarta_internasional', 
    'ngurah_rai_bali_internasional', 'juanda_surabaya_internasional', 
    'kualanamu_medan_internasional'
]

window_range = range(2, 25)
n_forecast = 12

for col in bandara_cols:
    print(f"\nANALYZING MA WINDOW: {col.upper()}")
    data = df[col].values
    
    split_idx = int(len(data) * 0.8)
    train, test = list(data[:split_idx]), list(data[split_idx:])
    
    ma_results = []
    
    for w in window_range:
        try:
            
            temp_train = train.copy()
            preds = []
            for i in range(len(test)):
            
                m = np.mean(temp_train[-w:])
                preds.append(m)
                
                temp_train.append(test[i])
            
            preds = np.array(preds)
            actuals = np.array(test)
            
            mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
            r2 = r2_score(actuals, preds)
            
            ma_results.append({
                'Window (k)': w,
                'R2': round(r2, 4),
                'MAPE (%)': round(mape, 2)
            })
        except: continue
    
    df_res = pd.DataFrame(ma_results).sort_values('MAPE (%)', ascending=True).head(5)
    display(df_res)


ANALYZING MA WINDOW: SOEKARNO_HATTA_JAKARTA_DOMESTIK


,Window (k),R2,MAPE (%)
0,2,0.2406,8.03
1,3,0.2021,8.73
2,4,0.1052,8.92
22,24,-0.0476,9.30
17,19,-0.0529,9.46



ANALYZING MA WINDOW: NGURAH_RAI_BALI_DOMESTIK


,Window (k),R2,MAPE (%)
11,13,0.0638,10.51
12,14,0.0546,10.55
10,12,0.0285,10.64
13,15,0.0131,10.84
9,11,-0.0852,11.09



ANALYZING MA WINDOW: JUANDA_SURABAYA_DOMESTIK


...

,Window (k),R2,MAPE (%)
0,2,0.5602,17.83
1,3,0.4333,21.05
2,4,0.3275,22.20
3,5,0.2104,24.46
4,6,0.1056,25.83



ANALYZING MA WINDOW: JUANDA_SURABAYA_INTERNASIONAL


,Window (k),R2,MAPE (%)
13,15,0.0608,8.02
14,16,0.0168,8.02
12,14,0.0498,8.12
11,13,0.0309,8.31
2,4,-0.2087,8.34



ANALYZING MA WINDOW: KUALANAMU_MEDAN_INTERNASIONAL


,Window (k),R2,MAPE (%)
6,8,-0.1583,6.81
4,6,-0.2320,6.89
5,7,-0.1965,6.91
3,5,-0.2706,6.98
7,9,-0.3040,7.25
